# മുൻഗണനാ അംഗം മിഡിൽവെയർ ഉപയോഗിയ്ക്കുന്ന ഹോട്ടൽ ബുക്കിംഗ്

ഈ നോട്ട്ബുക്ക് Microsoft Agent Framework ഉപയോഗിച്ച് **ഫംഗ്ഷൻ അടിസ്ഥാനമാക്കിയ മിഡിൽവെയർ** പ്രദർശിപ്പിക്കുന്നു. നാം കൺഡീഷണൽ വർക്ക്ഫ്ലോ ഉദാഹരണത്തെ അടിസ്ഥാനമാക്കി മുൻഗണനാ അംഗങ്ങൾക്ക് പ്രത്യേക സ്വീകരണങ്ങൾ നൽകുന്ന ഒരു മിഡിൽവെയർ ലെയർ ചേർക്കുന്നു.

## നിങ്ങൾ അറിയാനുള്ള കാര്യങ്ങൾ:
1. **ഫംഗ്ഷൻ-അടിസ്ഥാനമാക്കിയ മിഡിൽവെയർ**: ഫംഗ്ഷൻ ഫലങ്ങൾ തടയുകയും മാറ്റുകയും ചെയ്യുക  
2. **കോൺടക്സ് ആക്സസ്**: നിർവഹണത്തിനുശേഷം `context.result` വായിക്കുകയും മാറ്റുകയും ചെയ്യുക  
3. **ബിസിനസ് ലജിക് നടപ്പാക്കൽ**: മുൻഗണനാ അംഗങ്ങളുടെ പ്രയോജനങ്ങൾ  
4. **ഫലം തള്ളികാട്ടൽ**: ഉപയോക്തൃ നിലയുടെ അടിസ്ഥാനത്തിൽ ഫംഗ്ഷൻ ഫലം മാറ്റുക  
5. **അതെ വർക്ക്ഫ്ലോ, വ്യത്യസ്ത ഫലങ്ങൾ**: മിഡിൽവെയർ നയിക്കുന്ന പെരുമാറ്റ മാറ്റങ്ങൾ  

## മിഡിൽവെയർ ഉള്ള വർക്ക്ഫ്ലോ ആർക്കിടെക്ചർ:

```
User Input: "I want to book a hotel in Paris"
                    ↓
        [availability_agent]
        - Calls hotel_booking tool
        - 🌟 priority_check middleware intercepts
        - Checks user membership status
        - IF priority + no rooms → Override to available!
        - Returns BookingCheckResult
                    ↓
        Conditional Routing
           /                    \
    [has_availability]    [no_availability]
          ↓                      ↓
    [booking_agent]        [alternative_agent]
    (Priority override!)   (Regular users)
          ↓                      ↓
       [display_result executor]
```

## കൺഡീഷണൽ വർക്ക്ഫ്ലോയിൽ നിന്നുള്ള പ്രധാന വ്യത്യാസം:

**മിഡിൽവെയർ ഇല്ലാതെയുള്ളത്** (14-conditional-workflow.ipynb):  
- പാരിസ് ನಲ್ಲಿ മുറികൾ ഇല്ല → alternative_agent ലേക്ക് റൂട്ടുചെയ്യുക  

**മിഡിൽവെയർ ഉപയോഗിച്ചിരിക്കുന്നത്** (ഈ നോട്ട്ബുക്ക്):  
- സാധാരണ ഉപയോക്താവ് + പാരിസ് → മുറികൾ ഇല്ല → alternative_agent ലേക്ക് റൂട്ടുചെയ്യുക  
- മുൻഗണനാ ഉപയോക്താവ് + പാരിസ് → 🌟 മിഡിൽവെയർ പണിയെടുക്കുന്നു! → ലഭ്യമാണ് → booking_agent ലേക്ക് റൂട്ടുചെയ്യുക  

## മുൻപോർക്കുന്ന കാര്യങ്ങൾ:  
- Microsoft Agent Framework ഇൻസ്റ്റാൾ ചെയ്‌തിട്ടുള്ളത്  
- കൺഡീഷണൽ വർക്ക്ഫ്ലോകളുടെ വിശദാംശങ്ങൾ അറിയുക (14-conditional-workflow.ipynb കാണുക)  
- GitHub ടോക്കൺ അല്ലെങ്കിൽ OpenAI API കീ  
- മിഡിൽവെയർ പാറ്റേണുകളുടെ അടിസ്ഥാന ആശയം


In [1]:
import asyncio
import json
import os
from collections.abc import Awaitable, Callable
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    ChatMessage,
    FunctionInvocationContext,
    Role,
    WorkflowBuilder,
    WorkflowContext,
    ai_function,
    executor,
)

# 🤖 GitHub Models or OpenAI client integration
from agent_framework.openai import OpenAIChatClient
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")

✅ All imports successful!


## Step 1: ഘടനാപരമായ ഔട്ട്പുട്ടുകൾക്കായി പൈഡാന്റിക് മോഡലുകൾ നിർവ്വചിക്കുക

ഈ മോഡലുകൾ ഏജന്റുകൾ തിരികെ നൽകുന്ന **സ്‌കീമ** നിർവ്വചിക്കുന്നു. മിഡിൽവെയർ ലഭ്യത ഫലം മാറ്റുമ്പോൾ നിരീക്ഷിക്കാൻ `priority_override` ഫീൽഡ് കൂട്ടിച്ചേർത്തിട്ടുണ്ട്.


In [2]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str
    priority_override: bool = False  # 🆕 NEW! Tracks if middleware overrode the result


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check with priority_override)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

✅ Pydantic models defined:
   - BookingCheckResult (availability check with priority_override)
   - AlternativeResult (alternative suggestion)
   - BookingConfirmation (booking confirmation)


## Step 2: മുൻഗണന അംഗങ്ങളുടെ ഡാറ്റാബേസ് നിർവ്വചിക്കുക

ഈ ഡെമോയ്ക്ക്, നാം മുൻഗണന അംഗത്വ ഡാറ്റാബേസ് സിമുലേറ്റ് ചെയ്യും. പ്രൊഡക്ഷനിൽ, ഇത് യഥാർത്ഥ ഡാറ്റാബേസ് അല്ലെങ്കിൽ API കണ്ടെടുത്തുകേൾക്കും.

**മുൻഗണന അംഗങ്ങൾ:**
- `alice@example.com` - വിഐപി അംഗം
- `bob@example.com` - പ്രീമിയം അംഗം  
- `priority_user` - ടെസ്റ്റ് അക്കൗണ്ട്


In [3]:
# Simulated priority members database
PRIORITY_MEMBERS = {
    "alice@example.com",
    "bob@example.com",
    "priority_user",
}

# Global variable to track current user (in real app, use proper session management)
current_user_id = "regular_user"  # Default: regular user


def set_user(user_id: str):
    """Set the current user for the session."""
    global current_user_id
    current_user_id = user_id
    is_priority = user_id in PRIORITY_MEMBERS
    status = "🌟 PRIORITY MEMBER" if is_priority else "👤 Regular User"

    display(
        HTML(f"""
        <div style='padding: 15px; background: {"linear-gradient(135deg, #FFD700 0%, #FFA500 100%)" if is_priority else "#e3f2fd"}; 
                    border-left: 4px solid {"#FF6B35" if is_priority else "#2196f3"}; border-radius: 4px; margin: 10px 0;'>
            <strong>👤 Current User Set:</strong> {user_id}<br>
            <strong>Status:</strong> {status}
        </div>
    """)
    )


print("✅ Priority members database created")
print(f"   Priority members: {len(PRIORITY_MEMBERS)} users")

✅ Priority members database created
   Priority members: 3 users


## Step 3: ഹോട്ടല്‍ ബുക്കിംഗ് ടൂൾ സൃഷ്‌ടിക്കുക

കണ്ടീഷണൽ വെർkmen്ф്ലോ പോലെയാണ്, പക്ഷെ ഇപ്പോള്‍ ഇത് മിഡിൽവെയർ വഴി ഇടപെടപ്പെടുകയും ചെയ്യും!


In [4]:
@ai_function(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.
    
    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @ai_function decorator")

✅ hotel_booking tool created with @ai_function decorator


## Step 4: 🌟 പ്രാധാന്യ പരിശോധന മിഡിൽവെയർ സൃഷ്ടിക്കുക (പ്രധാന സവിശേഷത!)

ഈ നോൺബുക്കിന്റെ **മുൻനിര പ്രവർത്തനക്ഷമത** ഇതാണ്. മിഡിൽവെയർ:

1. ഹോട്ടൽ_ബുക്കിംഗ് ഫംഗ്ഷൻ കോൾ **അടിച്ചുപൊക്കി**
2. `next(context)` വിളിച്ച് ഫംഗ്ഷൻ സാധാരണ പ്രവർത്തിപ്പിക്കുന്നു
3. `context.result` ൽ ഫലം **പരിശോധിക്കുന്നു**
4. ഉപയോക്താവ് പ്രാധാന്യമുള്ളവരും മുറികൾ ലഭ്യമല്ലാത്തവരും ആണെങ്കിൽ ഫലം **മാറ്റി എഴുതുന്നു**
5. മാറ്റം വന്ന ഫലം ഏജന്റിന് **തിരികെ നൽകുന്നു**

**പ്രധാന മാതൃക:**
```python
async def my_middleware(context, next):
    await next(context)  # ഫംഗ്ഷൻ നിർവഹിക്കുക
    # ഇപ്പോള്‍ context.result ഫംഗ്ഷന്റെ ഔട്ട്പുട്ടിനെ അടങ്ങിയിരിക്കുന്നു
    if some_condition:
        context.result = new_value  # മേൽലേഖനം!
```


In [5]:
async def priority_check_middleware(
    context: FunctionInvocationContext,
    next: Callable[[FunctionInvocationContext], Awaitable[None]],
) -> None:
    """
    Function middleware that overrides hotel_booking results for priority members.
    
    Workflow:
    1. Let the function execute normally
    2. Check if user is a priority member
    3. If priority + no availability → Override to make rooms available!
    4. Agent will then route to booking path instead of alternative path
    """
    function_name = context.function.name

    display(
        HTML(f"""
        <div style='padding: 12px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
            <strong>🔄 Middleware:</strong> Intercepting {function_name}...
        </div>
    """)
    )

    # Execute the original function
    await next(context)

    # Now inspect and potentially modify the result
    if context.result and function_name == "hotel_booking":
        result_data = json.loads(context.result)
        destination = result_data.get("destination", "")
        has_availability = result_data.get("has_availability", False)

        # Check if user is priority member
        is_priority = current_user_id in PRIORITY_MEMBERS

        # Override logic: Priority member + no availability → Make available!
        if is_priority and not has_availability:
            display(
                HTML(f"""
                <div style='padding: 20px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); 
                            border-radius: 8px; margin: 10px 0; box-shadow: 0 4px 12px rgba(255,165,0,0.4);'>
                    <h3 style='margin: 0 0 10px 0; color: #333;'>🌟 PRIORITY OVERRIDE ACTIVATED! 🌟</h3>
                    <p style='margin: 0; color: #555; line-height: 1.6;'>
                        <strong>User:</strong> {current_user_id}<br>
                        <strong>Status:</strong> VIP Priority Member<br>
                        <strong>Action:</strong> Overriding "No Availability" for {destination}<br>
                        <strong>Result:</strong> ✅ Rooms now available for priority booking!
                    </p>
                </div>
            """)
            )

            # Override the result!
            result_data["has_availability"] = True
            result_data["priority_override"] = True
            context.result = json.dumps(result_data)

        elif not has_availability:
            display(
                HTML(f"""
                <div style='padding: 12px; background: #ffebee; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>ℹ️ Middleware:</strong> No priority override (user: {current_user_id})
                </div>
            """)
            )


print("✅ priority_check_middleware created")
print("   - Intercepts hotel_booking function")
print("   - Overrides availability for priority members")

✅ priority_check_middleware created
   - Intercepts hotel_booking function
   - Overrides availability for priority members


## Step 5: റൂട്ടിംഗ്‌ക്കായി നിബന്ധനാ ഫังก്ഷനുകൾ നിർവചിക്കുക

നിബന്ധനയായ വർക്ക്‌ഫ്ലോയിലെ പോലെ തന്നെയാണ് നിബന്ധനാ ഫังก്ഷനുകൾ - റൂട്ടിംഗ് നിർവചിക്കാൻ ഘടിതമായ ഔട്ട്പുട്ടിനെ അവ പരിശോധിക്കുന്നു.


In [6]:
def has_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels ARE available (including priority overrides!)."""
    if not isinstance(message, AgentExecutorResponse):
        return True

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        # Check if this was a priority override
        override_indicator = " 🌟" if result.priority_override else ""

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}{override_indicator}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels are NOT available."""
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception:
        return False


print("✅ Condition functions defined")

✅ Condition functions defined


## Step 6: കസ്റ്റം ഡിസ്പ്ലേ എക്സിക്യൂട്ടർ സൃഷ്ടിക്കുക

മുന്നത്തെ/executor പോലെ തന്നെ - അന്തിമ വർਕ്ഫ്ലോ-Ausgang് പ്രദർശിപ്പിക്കുന്നു.


In [7]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """Display the final result as workflow output."""
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created")

✅ display_result executor created


## Step 7: ലോഡ് ചെയ്യുന്ന അന്തരീക്ഷ വ്യത്യാസങ്ങൾ

LLM ക്ലയന്റ് (GitHub മോഡലുകൾ അല്ലെങ്കിൽ OpenAI) കോൺഫിഗർ ചെയ്യുക.


In [8]:
# Load environment variables
load_dotenv()

# Check for GitHub Models or OpenAI
chat_client = OpenAIChatClient(base_url=os.environ.get(
    "GITHUB_ENDPOINT"), api_key=os.environ.get("GITHUB_TOKEN"), model_id="gpt-4o")


## പടി 8: മിഡിൽവെയർ ഉപയോഗിച്ച് AI ഏജന്റുകൾ സൃഷ്ടിക്കുക

**പ്രധാന വ്യത്യാസം:** availability_agent സൃഷ്ടിക്കുമ്പോൾ, നാം `middleware` പാരാമീറ്റർ നൽകുന്നു!

ഇതാണ് priority_check_middleware ഏജന്റിന്റെ ഫങ്ക്ഷൻInvocation പൈപ്പ്ലൈനിലേക്കുള്ള ഇഴശിപ്പിക്കൽ.


In [9]:
# Agent 1: Check availability with tool + middleware
availability_agent = AgentExecutor(
    chat_client.create_agent(
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), message (string), "
            "and priority_override (bool, true if priority member got special access). "
            "The message should summarize the availability status and mention if priority override occurred."
        ),
        tools=[hotel_booking],
        response_format=BookingCheckResult,
        middleware=[priority_check_middleware],  # 🌟 MIDDLEWARE INJECTION!
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    chat_client.create_agent(
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        response_format=AlternativeResult,
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    chat_client.create_agent(
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "If priority_override is true in the input, mention that they received priority member access. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        response_format=BookingConfirmation,
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - WITH priority_check_middleware 🌟</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)

## ഘട്ടം 9: വർക്ക്‌ഫ്ലോ നിർമ്മിക്കുക

മുമ്പത്തെ പോലെ തന്നെ വർക്ക്‌ഫ്ലോ രൂപരേഖ - ലഭ്യതയുടെ അടിസ്ഥാനത്തിൽ ശ്രതീയ ഓരോത്.


In [10]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder()
    .set_start_executor(availability_agent)
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH (can be triggered by middleware override!)
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing (Middleware-Aware):</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> (or 🌟 <strong>priority override</strong>) → booking_agent → display_result
        </p>
    </div>
""")
)

## Step 10: ടെസ്റ്റ് കേസ് 1 - പാരിസിലെ സാധാരണ ഉപയോക്താവ് (ഓവർറൈഡ് ഇല്ലാതെ)

ഒരു സാധാരണ ഉപയോക്താവ് പാരിസിലേക്ക് ബുക്ക് ചെയ്യാൻ ശ്രമിക്കുന്നു → മുറികൾ ഇല്ല → alternative_agent-ലേക്ക് റൂട്ടുചെയ്യുന്നു


In [11]:
# Set as regular user
set_user("regular_user")

display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Regular User + Paris</h3>
        <p style='margin: 0;'><strong>Expected:</strong> No rooms → No middleware override → Alternative suggestion</p>
    </div>
""")
)

# Create request
request_regular = AgentExecutorRequest(
    messages=[ChatMessage(Role.USER, text="I want to book a hotel in Paris")], should_respond=True
)

# Run workflow
events_regular = await workflow.run(request_regular)
outputs_regular = events_regular.get_outputs()

# Display results
if outputs_regular:
    result_regular = AlternativeResult.model_validate_json(outputs_regular[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: #fff; border: 2px solid #ff9800; border-radius: 12px; margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #e65100;'>📊 RESULT (Regular User)</h3>
            <div style='background: #fff3e0; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0;'><strong>Middleware:</strong> No priority override (regular user)</p>
                <p style='margin: 0 0 10px 0;'><strong>Alternative:</strong> 🏨 {result_regular.alternative_destination}</p>
                <p style='margin: 0;'><strong>Reason:</strong> {result_regular.reason}</p>
            </div>
        </div>
    """)
    )

## Step 11: Test Case 2 - 🌟 പ്രാഥമിക ഉപഭോക്താവ് പാരിസിൽ (ഒവർറൈഡ് ഉള്ളത്!)

ഒരു പ്രാധാന്യമുള്ള അംഗം പാരിസിലേക്ക് ബുക്ക് ചെയ്യാൻ ശ്രമിക്കുന്നു → ആദ്യം മുറികൾ ഒന്നും ലഭ്യമല്ല → 🌟 മിഡിൽവെയർ ഒവർറൈഡ് ചെയ്യുന്നു! → booking_agent ലേക്ക് റൂട്ടുചെയ്യുന്നു

**ഇത് മിഡിൽവെയറിന്റെ ശക്തിയുടെ മുഖ്യ പ്രദർശനമാണ്!**


In [12]:
# Set as priority user
set_user("priority_user")

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #333;'>🧪 TEST CASE 2: 🌟 Priority User + Paris</h3>
        <p style='margin: 0; color: #555;'><strong>Expected:</strong> No rooms → 🌟 MIDDLEWARE OVERRIDE → Rooms available → Booking suggestion!</p>
    </div>
""")
)

# Create request
request_priority = AgentExecutorRequest(
    messages=[ChatMessage(Role.USER, text="I want to book a hotel in Paris")], should_respond=True
)

# Run workflow
events_priority = await workflow.run(request_priority)
outputs_priority = events_priority.get_outputs()

# Display results
if outputs_priority:
    result_priority = BookingConfirmation.model_validate_json(outputs_priority[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; 
                    box-shadow: 0 8px 16px rgba(255,165,0,0.4); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 RESULT (Priority Member) 🌟</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available (Priority Override!)</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Middleware:</strong> 🌟 OVERRIDE ACTIVATED!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_priority.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_priority.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_priority.message}</p>
                <div style='margin-top: 15px; padding: 15px; background: #fff3cd; border-radius: 6px; border-left: 4px solid #FF6B35;'>
                    <strong>💡 What Just Happened:</strong><br>
                    1. hotel_booking tool returned "no availability"<br>
                    2. priority_check_middleware intercepted the result<br>
                    3. Middleware checked user status: priority_user ✅<br>
                    4. Middleware OVERRODE the result to "available"<br>
                    5. Workflow routed to booking_agent instead of alternative_agent!
                </div>
            </div>
        </div>
    """)
    )

## ഘട്ടം 12: ടെസ്റ്റ് കേസ് 3 - സ്റ്റോക്ക്ഹോംയിലെ പ്രാഥമിക ഉപഭോക്താവ് (ഇതിനകം ലഭ്യമാണ്)

പ്രാഥമിക ഉപഭോക്താവ് സ്റ്റോക്ക്ഹോം പരീക്ഷിക്കുന്നു → മുറികൾ ലഭ്യമാണ് → ഓവർറൈഡ് ആവശ്യമില്ല → booking_agent ലേക്ക്റൂട്ടുചെയ്യുന്നു

ഇത് കാണിക്കുന്നത് മിഡിൽവെയർ ആവശ്യമായപ്പോൾ മാത്രമേ പ്രവർത്തിക്കുകയുള്ളൂ എന്നതാണ്!


In [13]:
# Priority user is still set from previous test

display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 3: Priority User + Stockholm</h3>
        <p style='margin: 0;'><strong>Expected:</strong> Rooms available → No override needed → Booking suggestion</p>
    </div>
""")
)

# Create request
request_stockholm = AgentExecutorRequest(
    messages=[ChatMessage(Role.USER, text="I want to book a hotel in Stockholm")], should_respond=True
)

# Run workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; 
                    box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 RESULT (Priority User - No Override Needed)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available (Natural)</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Middleware:</strong> No override needed</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
                <div style='margin-top: 15px; padding: 15px; background: #e8f5e9; border-radius: 6px; border-left: 4px solid #4caf50;'>
                    <strong>💡 Middleware Behavior:</strong><br>
                    • hotel_booking returned "available" naturally<br>
                    • Middleware saw available = true → No override needed<br>
                    • Workflow proceeded normally to booking_agent
                </div>
            </div>
        </div>
    """)
    )

## പ്രധാന ആശയങ്ങൾ ಮತ್ತು മിഡിൽവെയർ ആശയങ്ങൾ

### ✅ നിങ്ങൾ പഠിച്ചിരിക്കുന്നത്:

#### **1. ഫംഗ്ഷന്-അടിസ്ഥിതമായ മിഡിൽവെയർ പാറ്റേൺ**

മിഡിൽവെയർ ഒരു ലളിതമായ അസിങ്ക് ഫംഗ്ഷൻ ഉപയോഗിച്ച് ഫംഗ്ഷൻ കോൾ interception ചെയ്യുന്നു:

```python
async def my_middleware(
    context: FunctionInvocationContext,
    next: Callable,
) -> None:
    # ഫംഗ്ഷൻ പ്രവർത്തനം തുടങ്ങുന്നതിന് മുമ്പ്
    print("Intercepting...")
    
    # ഫംഗ്ഷൻ പ്രവർത്തിപ്പിക്കുക
    await next(context)
    
    # ഫംഗ്ഷൻ പ്രവർത്തനം കഴിഞ്ഞ് - ഫലം പരിശോധിക്കുക
    if context.result:
        # ആവശ്യമെങ്കിൽ ഫലം മാറ്റം വരുത്തുക
        context.result = modified_value
```

#### **2. കോൺടെക്സ്‌ട് ആക്സസ് ചെയ്യൽ மற்றும் ഫലം ओവർറൈഡ് ചെയ്യൽ**

- `context.function` - വിളിക്കപ്പെടുന്ന ഫംഗ്ഷൻ ആക്സസ് ചെയ്യുക
- `context.arguments` - ഫംഗ്ഷൻ ആർഗുമെന്റുകൾ വായിക്കുക
- `context.kwargs` - അധിക പാരാമീറ്ററുകൾ ആക്സസ് ചെയ്യുക
- `await next(context)` - ഫംഗ്ഷൻ നടത്തുക
- `context.result` - ഫംഗ്ഷന്റെ ഔട്ട്‌പുട്ട് വായിക്കുക/പരിഷ്കരിക്കുക

#### **3. ബിസിനസ് ലജിക് നടപ്പാക്കൽ**

നമ്മുടെ മിഡിൽവെയർ പ്രിയോറിറ്റി മെംബർ ബിനിഫിറ്റുകൾ നടപ്പാക്കുന്നു:
- **പതിവ് ഉപയോക്താക്കൾ**: മാറ്റങ്ങൾ ഇല്ല, സാധാരണ വർക്ക്‌ഫ്ലോ
- **പ്രിയോറിറ്റി ഉപയോക്താക്കൾ**: "അവൈലബിലിറ്റി ഇല്ല" → "ലഭ്യമാണ്" ഒവർറൈഡ് ചെയ്യുന്നു
- **ശർത്തി അടിസ്ഥാനത്തിലുള്ള ലജിക്**: ആവശ്യം വരുമ്പോഴേ ഒവർറൈഡ് ചെയ്യുക

#### **4. സമാന വർക്ക്‌ഫ്ലോ, വ്യത്യസ്ത ഫലം**

മിഡിൽവെയറിന്‍റെ ശക്തി:
- ✅ വർക്ക്‌ഫ്ലോ ഘടനയിൽ മാറ്റങ്ങൾ ഇല്ല
- ✅ ടൂൾ ഫംഗ്ഷനിൽ മാറ്റങ്ങൾ ഇല്ല
- ✅ ശരConditions routing ലജിക് മാറ്റം ഇല്ല
- ✅ വെറും മിഡിൽവെയർ → വ്യത്യസ്ത പെരുമാറ്റം!

### 🚀 യഥാർത്ഥ ലോക പ്രയോഗങ്ങൾ:

1. **VIP/പ്രീമിയം ഫീച്ചറുകൾ**
   - പ്രീമിയം ഉപയോക്താക്കൾക്ക് റേറ്റ് ലിമിറ്റുകൾ ഒവർറൈഡ് ചെയ്യുക
   - റിസോഴ്‌സുകൾക്ക് പ്രിയോറിറ്റി ആക്സസ് നൽകുക
   - പ്രീമിയം ഫീച്ചറുകൾ ഡൈനാമിക്കായി തുറക്കുക

2. **A/B ടെസ്റ്റിംഗ്**
   - ഉപയോക്താക്കളെ വ്യത്യസ്ത നടപ്പിലാക്കിയ ഭാഗങ്ങളിലേക്ക് റൂട്ടുചെയ്‌ത്
   - പ്രത്യേക ഉപയോക്താക്കളുമായി പുതിയ ഫീച്ചറുകൾ പരീക്ഷിക്കുക
   - ക്രമത്തിൽ ഫീച്ചർ റിലീസുകൾ

3. **സുരക്ഷ & അനുസരണം**
   - ഫംഗ്ഷൻ കോൾസ് ഓഡിറ്റ് ചെയ്യുക
   - സेंसിറ്റീവ് ഓപ്പറേഷനുകൾ ബ്ലോക്ക് ചെയ്യുക
   - ബിസിനസ് റൂൾസ് നടപ്പാക്കുക

4. **പ്രകടന മെച്ചപ്പെടുത്തൽ**
   - പ്രത്യേക ഉപയോക്താക്കൾക്ക് ഫലം കാഷ് ചെയ്യുക
   - എത്രയും വൈകാതെ ഓപ്പറേഷനുകൾ ഒഴിവാക്കുക
   - ഡൈനാമിക് റിസോഴ്‌സ് അലോക്കേഷൻ

5. **പിശക് കൈകാര്യം & റിട്രൈ**
   - പിശകുകൾ പിടിച്ചെടുത്തു സ്നേഹപൂർവ്വം കൈകാര്യം ചെയ്യുക
   - റിട്രൈ ലജിക് നടപ്പാക്കുക
   - നടത്തുന്ന ക്ലീസായ നിർവഹണങ്ങൾക്കായി ഫാൾബാക്ക്

6. **ലോഗിംഗ് & മോണിറ്ററിംഗ്**
   - ഫംഗ്ഷൻ പ്രവർത്തന സമയം ട്രാക്ക് ചെയ്യുക
   - പാരാമീറ്ററുകളും ഫലങ്ങളും ലോഗ് ചെയ്യുക
   - ഉപയോക്തൃ ഉപയോഗ പാറ്റേണുകൾ നിരീക്ഷിക്കുക

### 🔑 ഡിസ്ക്രിപ്ഷനുകൾ ഡെക്കറേറ്ററുകളുമായി താരതമ്യം ചെയ്യുമ്പോൾ:

| ഫീച്ചർ | ഡെക്കറേറ്റർ | മിഡിൽവെയർ |
|---------|-----------|------------|
| **പരിധി** | ഏകീകൃത ഫംഗ്ഷൻ | ഏജന്റ്中的 എല്ലാ ഫംഗ്ഷനുകളും |
| **പ്രവർത്തനക്ഷമത** | നിർവ്വചിച്ചപ്പോൾ നിശ്ചിതം | പ്രവർത്തന സമയത്ത് ഡൈനാമിക് |
| **കോൺടെക്സ്റ്റ്** | പരിമിതം | മുഴുവൻ ഏജന്റ് കോൺടെക്സ്റ്റ് |
| **സംയോജനം** | പല ഡെക്കറേറ്ററുകൾ | മിഡിൽവെയർ പൈപ്പ്ലൈൻ |
| **ഏജന്റ്-അവേർ** | ഇല്ല | ഉണ്ട് (ഏജന്റ് സ്റ്റേറ്റിലേക്ക് ആക്സസ്) |

### 📚 മിഡിൽവെയർ ഉപയോഗിക്കാനുള്ള സമയത്ത്:

✅ **മിഡിൽവെയർ ഉപയോഗിക്കുക:**
- ഉപയോക്താവ്/സെഷൻ സ്റ്റേറ്റിന്റെ അടിസ്ഥാനത്തിൽ പെരുമാറ്റം മാറ്റേണ്ടതുണ്ടെങ്കിൽ
- പല ഫംഗ്ഷനുകൾക്കും ലജിക് ബാധിക്കണമെന്ന് ആഗ്രഹിക്കുമ്പോൾ
- ഏജന്റ്-ലവൽ കോൺടെക്സ്റ്റ് ആക്സസ് ആവശ്യമുള്ളപ്പോൾ
- ക്രോസ്-കട്ടിംഗ് ആശങ്കകൾ (ലോഗിംഗ്, അഥോറൈസേഷൻ, മുതലായവ) നടപ്പാക്കുകയാണെങ്കിൽ

❌ **മിഡിൽവെയർ ഉപയോഗിക്കേണ്ടതില്ല:**
- ലളിതമായ ഇൻപുട്ട് ഒപ്പം പരിശോധന (പൈഡാന്റിക് ഉപയോഗിക്കുക)
- ഫംഗ്ഷൻ-നിർദിഷ്ടമായ ലജിക് (ഫംഗ്ഷനിലും സൂക്ഷിക്കുക)
- ഒറ്റത്തവണ മാറ്റങ്ങൾ (വേറെ ചെയ്യാതെ ഫംഗ്ഷൻ തന്നെ മാറ്റുക)

### 🎓 പ്രഗത്ഭമായ പാറ്റേണുകൾ:

```python
# ഒരേ സമയം എക്സിക്യൂഷൻ ഓർഡർ പ്രധാനമാണ് ഉള്ള പല മിഡിൽവെയർများ!
middleware=[
    logging_middleware,      # ആദ്യം ലോഗുകൾ
    auth_middleware,         # പിന്നീട് authentication പരിശോധിക്കുന്നു
    cache_middleware,        # ശേഷം കാഷെ പരിശോധിക്കുന്നു
    rate_limit_middleware,   # പിന്നീട് റേറ്റ് ലിമിറ്റുകൾ പ്രയോഗിക്കുന്നു
    priority_check_middleware  # അവസാനം പ്രാധാന്യ പരിശോധന
]

# നിബന്ധന അടിസ്ഥാനമായ മിഡിൽവെയർ എക്സിക്യൂഷൻ
async def conditional_middleware(context, next):
    if should_execute(context):
        await next(context)
        # ഫലം മാറ്റം വരുത്തുക
    else:
        # മുഴുവനും എക്സിക്യൂഷൻ ഒഴിവാക്കുക
        context.result = cached_value
```

### 🔗 ബന്ധപ്പെട്ട ആശയങ്ങൾ:

- **ഏജന്റ് മിഡിൽവെയർ**: <code>agent.run()</code> കോൾ interceptions
- **ഫംഗ്ഷൻ മിഡിൽവെയർ**: ടൂൾ ഫംഗ്ഷൻ കോൾ interception (നമ്മൾ ഉപയോഗിച്ചത്!)
- **മിഡിൽവെയർ പൈപ്പ്ലൈൻ**: മിഡിൽവെയറുകളുടെ ചങ്ങല ക്രമഭദ്രമായി പ്രവർത്തിക്കുന്നു
- **കോൺടെക്സ്‌റ്റ് പ്രൊപ്പഗേഷൻ**: സ്റ്റേറ്റ് മിഡിൽവെയർ ചങ്ങല വഴി കടത്തുക


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**പരിമിതീകരണം**:  
ഈ ഡോക്യുമെന്റ് എ.ഐ. വിവർത്തനസേവനായ [കോ-ഓപ് ട്രാൻസ്ലേറ്റർ](https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് വിവർത്തനം ചെയ്തതാണ്. നാം കൃത്യതയ്‌ക്കായി പരിശ്രമിച്ചെങ്കിലും, ഓട്ടോമാറ്റഡ് വിവർത്തനങ്ങളിൽ പിശകുകൾ അല്ലെങ്കിൽ അയോഗ്യതകൾ ഉണ്ടാകാമെന്ന കാര്യം ശ്രദ്ധിക്കണം. പ്രത്യേക ഭാഷയിൽ ഉള്ള മൂല ഡോക്യുമെന്റും അതാണ് പ്രാധാന്യമുള്ള ഉറവിടം. നിർണായക വിവരങ്ങൾക്ക്, പ്രൊഫഷണൽ മനുഷ്യ വിവർത്തനം ശുപാർശ ചെയ്യപ്പെടുന്നു. ഈ വിവർത്തനം ഉപയോഗിക്കുന്നതിനാൽ ഉണ്ടാകുന്ന തെറ്റിദ്ധാരണകൾക്കും അർത്ഥക്കുഴപ്പങ്ങൾക്കും ഞങ്ങൾ ഉത്തരവാദികളല്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
